# Tabula Sapiens fat: cells, QC, clusters, endothelial cells, SAT vs VAT, and ERG

**Author's note (junior bioinformatician).** This notebook takes the Tabula Sapiens fat file
(`Fat_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad`) and runs the plan from your guide
("Single Cell data analysis steps"):

`metadata audit -> ambient RNA -> doublets -> mitochondrial cut-off (15%) -> elbow plot -> clusters ->
annotate with DEGs + Human Protein Atlas markers -> endothelial cells -> their own elbow/UMAP/subtypes ->
SAT vs VAT -> ERG -> pseudobulk (control vs T2D) -> bar graph of the 8 genes`

## Read this first: one thing may not be possible
Tabula Sapiens is an atlas of **organ donors without a diabetes diagnosis**. I could not open your 4 GB file from my
workspace, so I cannot confirm what its metadata holds. **Section 3 audits it and prints the donors, the depot labels and
the disease labels. The control-vs-T2D tests only run if that audit finds both a control group and a T2D group with
at least 3 donors each.** If it does not, the notebook says so and does *not* invent a contrast. It then runs the
**SAT vs VAT** comparison (same donors, paired) so that you still get a defensible ERG result, clearly labelled as a depot
comparison and not a diabetes result. Your guide asks for exactly this: "if ERG is too weakly measured ... report
insufficient information rather than forcing a result".

## How your guide shaped the choices
| Guide point | What the notebook does |
|---|---|
| Donors, not cells, are the replicates | Counts are summed per donor (x depot x subtype) before any test |
| Keep raw counts; do not test on corrected values | Raw counts stay in assay `RAW`. Ambient-corrected counts are used for clustering and the pseudobulk is made from the same counts you choose in the settings |
| Do not drop ERG because it is not a variable gene | The panel genes are forced into every test |
| Do not require ERG positivity to select ECs | ECs are picked from clusters named by other markers |
| No universal "200 genes / 5% mito" rule | Cut-offs are shown per library before they are applied; you asked for **15% mito** |
| Do not regress out disease | Only `nCount_RNA` is regressed; integration is by donor |
| Keep an exclusion log, compare groups | Section 4 logs every removal per library and per group |
| A donor with no cells of a subtype is *missing*, not zero | Samples with fewer than 20 cells are dropped |
| Report insufficient information rather than force a result | Gated by donor numbers and read counts |

## Packages
`Seurat` (v5), `Matrix`, `edgeR`, `ggplot2`, `dplyr`. Optional but used when present: `celda` (DecontX), `scDblFinder`,
`DESeq2`, `presto` (faster markers). Python with `anndata` and `scipy` is needed once, to turn the .h5ad into plain files.
About 16 GB of memory is advisable for the 4 GB file.

*Caution:* the code was tested on small simulated data (both with and without a T2D group). Check every table against what
you expect before you trust a result.

## 1. Settings

**Cell annotation:** Everything you may want to change is here. The zip path is the one you gave; use forward slashes in R.

In [ ]:
h5ad_zip     <- "C:/Users/user/Downloads/TS_Fat.h5ad.zip"   # your zip (it holds the .h5ad)
export_dir   <- "TS_fat_export"          # plain files written from the .h5ad
out_dir      <- "TS_fat_results"         # all results go here
python_cmd   <- "python"                 # the python that has anndata + scipy installed

# --- quality control
max_mito_pct <- 15      # your cut-off: cells above this are removed
min_genes    <- 200     # loose floor, shown per library first (guide: inspect, do not apply blindly)
min_counts   <- 500
ambient      <- "auto"  # "auto": use the file's decontXcounts layer if present, else run DecontX; "decontx": always run DecontX; "none"
run_doublets <- TRUE    # scDblFinder, per library, on 10x cells only (smart-seq2 plates are not droplets)

# --- clustering
n_var_genes  <- 2000
res_all      <- 0.3     # resolution for all cells (the paper's 0.1 suits vascular-only data; raise it if clusters look merged)
res_ec       <- 0.7     # resolution for endothelial cells (paper's value)
max_pcs      <- 30      # most PCs used for neighbours/UMAP
min_cells_batch <- 100  # a donor with fewer cells is pooled with other small ones for integration
integrate    <- TRUE    # Seurat integration across donors (RPCA). Disease is never regressed out.

# --- annotation
hpa_markers_csv <- ""   # optional: your own table (columns cell_type, gene) exported from proteinatlas.org
min_score    <- 0.25    # a cluster needs this mean z-score for a label, and this margin over the runner-up
min_margin   <- 0.25

# --- statistics
panel_genes  <- c("ERG","PECAM1","CDH5","NOS3","VCAM1","ICAM1","HIF1A","ANGPT2")
panel_names  <- c(ERG="ERG", PECAM1="CD31 (PECAM1)", CDH5="VE-cadherin (CDH5)", NOS3="eNOS (NOS3)",
                  VCAM1="VCAM1", ICAM1="ICAM1", HIF1A="HIF1-alpha (HIF1A)", ANGPT2="ANGPT2")
min_cells_sample  <- 20 # fewer ECs than this in a donor x depot sample: the sample is left out (missing, not zero)
min_donors_group  <- 3  # donors needed in each group for a group test
fdr_cut      <- 0.05
set.seed(1)

**Cell annotation:** Load the packages and stop early, with a clear message, if one is missing.

In [ ]:
suppressPackageStartupMessages({ library(Seurat); library(Matrix); library(edgeR); library(ggplot2); library(dplyr) })
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
have <- function(p) requireNamespace(p, quietly = TRUE)
cat("Seurat", as.character(packageVersion("Seurat")), "| edgeR", as.character(packageVersion("edgeR")), "\n")
cat("optional: celda", have("celda"), "| scDblFinder", have("scDblFinder"), "| DESeq2", have("DESeq2"), "| presto", have("presto"), "\n")
if (as.integer(substr(as.character(packageVersion("Seurat")), 1, 1)) < 5) stop("Seurat 5 or later is needed.")

*What you should see:* Version lines. TRUE/FALSE for the optional packages (a FALSE only skips that optional step).

## 2. Unzip the file and turn the .h5ad into plain files

**Cell annotation:** The zip is unpacked into its own folder, then a small Python script (`scripts/export_TS_fat_h5ad.py`) writes the counts as a sparse matrix plus the cell and gene tables. Nothing is normalised. If you have already exported, this cell is skipped.

In [ ]:
exp_ok <- all(file.exists(file.path(export_dir, c("raw_counts.mtx.gz", "cells.csv", "genes.csv"))))
if (!exp_ok) {
  unz_dir <- file.path(dirname(export_dir), "TS_fat_unzipped"); dir.create(unz_dir, showWarnings = FALSE, recursive = TRUE)
  if (grepl("\\.zip$", h5ad_zip, ignore.case = TRUE)) { unzip(h5ad_zip, exdir = unz_dir) } else { unz_dir <- dirname(h5ad_zip) }
  h5ad <- list.files(unz_dir, pattern = "\\.h5ad$", recursive = TRUE, full.names = TRUE)
  if (length(h5ad) != 1) stop("Expected exactly one .h5ad, found: ", paste(h5ad, collapse = ", "))
  cat("h5ad:", h5ad, " size (GB):", round(file.size(h5ad) / 1e9, 2), "\n")
  print(tools::md5sum(h5ad))   # should be 8269b64d360e060b36941a1261fcea0c
  script <- "scripts/export_TS_fat_h5ad.py"
  if (!file.exists(script)) stop("Put export_TS_fat_h5ad.py in a 'scripts' folder next to this notebook.")
  rc <- system2(python_cmd, c(shQuote(script), shQuote(h5ad), shQuote(export_dir)))
  if (rc != 0) stop("The Python export failed. Run it in a terminal to see the error: python scripts/export_TS_fat_h5ad.py <h5ad> ", export_dir)
}
cat("layers in the file:", readLines(file.path(export_dir, "layers.txt")), "\n")
cat("raw counts taken from:", readLines(file.path(export_dir, "raw_counts_source.txt")), "\n")

*What you should see:* The md5 should equal 8269b64d360e060b36941a1261fcea0c (the checksum you were given). The layer list normally shows X, raw_counts, decontXcounts and others.

## 3. Cell information, the Seurat object, and the metadata audit
The guide (step 7) asks to confirm the matrix orientation, that every barcode maps to the metadata, that **ERG is in the
feature list**, and that donors, libraries and tissues are not confused. This section does that and then decides, from the
data, which tests are possible.

**Cell annotation:** Read the counts and cell table, check they line up, and build the object. Raw counts go in assay `RAW`; the working assay `RNA` starts as the same counts and is replaced by the ambient-corrected counts in section 4.

In [ ]:
mtx <- function(f) as(Matrix::readMM(file.path(export_dir, f)), "CsparseMatrix")
cells <- read.csv(file.path(export_dir, "cells.csv"), check.names = FALSE, stringsAsFactors = FALSE)
genes <- read.csv(file.path(export_dir, "genes.csv"), check.names = FALSE, stringsAsFactors = FALSE)
raw   <- mtx("raw_counts.mtx.gz")
stopifnot(ncol(raw) == nrow(cells), nrow(raw) == nrow(genes))        # orientation: genes x cells
# gene symbols as row names; duplicated symbols are made unique and the original Ensembl-style id is kept
sym <- make.unique(genes$symbol); rownames(raw) <- sym
colnames(raw) <- cells$cell_id; rownames(cells) <- cells$cell_id
gene_map <- data.frame(symbol = sym, feature_id = genes$feature_id)
write.csv(gene_map, file.path(out_dir, "gene_id_map.csv"), row.names = FALSE)
cat("cells:", ncol(raw), " genes:", nrow(raw), "\n")
cat("ERG present in the feature list:", "ERG" %in% rownames(raw), "\n")
cat("panel genes missing:", paste(setdiff(panel_genes, rownames(raw)), collapse = ", "), "(none = good)\n")
cat("counts are whole numbers:", all(raw@x[1:min(1e5, length(raw@x))] == round(raw@x[1:min(1e5, length(raw@x))])), "\n")
cat("\nColumns of the cell table:\n"); print(colnames(cells))

*What you should see:* Cells and genes, ERG TRUE, no missing panel genes, counts TRUE, then the list of metadata columns Tabula Sapiens provides.

**Cell annotation:** Find the columns for donor, method (10x or smart-seq2), depot, disease, sex and age. The first match in each list is used; **check the printout** and change the names if they are wrong.

In [ ]:
pick <- function(cands) { h <- cands[cands %in% colnames(cells)]; if (length(h)) h[1] else NA_character_ }
col_donor   <- pick(c("donor", "donor_id", "individual", "subject"))
col_method  <- pick(c("method", "assay", "technology"))
col_depot   <- pick(c("anatomical_information", "tissue_in_publication", "tissue", "anatomical_position", "sample_source"))
col_disease <- pick(c("disease", "disease_state", "condition", "diabetes", "t2d_status"))
col_sex     <- pick(c("sex", "gender"))
col_age     <- pick(c("age", "development_stage", "age_years", "donor_age"))
col_ct_ts   <- pick(c("cell_ontology_class", "free_annotation", "cell_type"))
cols <- c(donor = col_donor, method = col_method, depot = col_depot, disease = col_disease, sex = col_sex, age = col_age, TS_label = col_ct_ts)
print(cols)
if (is.na(col_donor)) stop("No donor column found. Donor-level analysis is impossible without it: name the column in col_donor.")
for (k in names(cols)) if (!is.na(cols[k])) { cat("\n--", k, "(", cols[k], ") --\n"); print(head(sort(table(cells[[cols[k]]], useNA = "ifany"), decreasing = TRUE), 15)) }

*What you should see:* A table of which column was used for what, then the counts of each value. Look especially at the depot and disease rows.

**Cell annotation:** Make clean working columns. **Depot:** text is matched with the patterns below (edit them if the Tabula Sapiens labels differ). **Disease:** values such as `normal`/`healthy` become Control; values that mention type 2 diabetes become T2D; anything else stays as its own label and is *not* used as a group.

In [ ]:
sat_regex <- "subcut|\\bSAT\\b|hypoderm|abdominal wall|inguinal|gluteal"
vat_regex <- "visceral|omental|\\bVAT\\b|mesenter|perirenal|intra-?abdominal|epididym"
ctl_regex <- "^normal$|healthy|control|^none$|non-?diabetic"
t2d_regex <- "type ?2|t2d|t2dm|diabet"

depot_txt <- if (!is.na(col_depot)) as.character(cells[[col_depot]]) else rep(NA, nrow(cells))
depot <- ifelse(grepl(sat_regex, depot_txt, ignore.case = TRUE), "SAT",
         ifelse(grepl(vat_regex, depot_txt, ignore.case = TRUE), "VAT", "Unknown"))
dis_txt <- if (!is.na(col_disease)) as.character(cells[[col_disease]]) else rep(NA, nrow(cells))
group <- ifelse(grepl(t2d_regex, dis_txt, ignore.case = TRUE), "T2D",
         ifelse(grepl(ctl_regex, dis_txt, ignore.case = TRUE), "Control", "Other/unknown"))

cells$donor   <- as.character(cells[[col_donor]])
cells$method  <- if (!is.na(col_method)) as.character(cells[[col_method]]) else "unknown"
cells$depot   <- depot
cells$group   <- group
cells$sex     <- if (!is.na(col_sex)) as.character(cells[[col_sex]]) else NA
cells$age_num <- if (!is.na(col_age)) suppressWarnings(as.numeric(gsub("[^0-9.]", "", as.character(cells[[col_age]])))) else NA
cells$library <- paste(cells$donor, cells$method, sep = "_")
cells$TS_label <- if (!is.na(col_ct_ts)) as.character(cells[[col_ct_ts]]) else NA

cat("Depot label -> depot (check the mapping):\n"); print(table(depot_txt, cells$depot, useNA = "ifany"))
cat("\nDisease label -> group:\n"); print(table(dis_txt, cells$group, useNA = "ifany"))
audit <- cells %>% count(donor, method, depot, group, sex, name = "cells") %>% arrange(donor)
print(as.data.frame(audit)); write.csv(audit, file.path(out_dir, "metadata_audit_donor_table.csv"), row.names = FALSE)

# --- which statistical questions the data can answer
donors_per_group <- cells %>% distinct(donor, group) %>% count(group)
n_ctl <- sum(donors_per_group$n[donors_per_group$group == "Control"]); n_t2d <- sum(donors_per_group$n[donors_per_group$group == "T2D"])
has_t2d   <- n_ctl >= min_donors_group && n_t2d >= min_donors_group
has_depot <- all(c("SAT", "VAT") %in% cells$depot)
donors_both <- cells %>% filter(depot %in% c("SAT","VAT")) %>% distinct(donor, depot) %>% count(donor) %>% filter(n == 2) %>% nrow()
cat("\nControl donors:", n_ctl, " T2D donors:", n_t2d, " -> control-vs-T2D test possible:", has_t2d, "\n")
cat("Both SAT and VAT labelled:", has_depot, " donors with both depots:", donors_both, "\n")
if (!has_t2d) message("NO usable T2D group (need >= ", min_donors_group, " donors in each of Control and T2D). The control-vs-T2D tests will be skipped; the SAT-vs-VAT comparison runs instead if the depots exist.")

*What you should see:* Two mapping tables, the donor table, and three lines saying which comparisons are possible. This is the decision point: read it before going on.

**Cell annotation:** Build the Seurat object. The original counts are kept in assay `RAW` and never modified.

In [ ]:
obj <- CreateSeuratObject(counts = raw, meta.data = cells[, setdiff(colnames(cells), "cell_id")], assay = "RNA", min.cells = 0, min.features = 0)
obj[["RAW"]] <- CreateAssay5Object(counts = raw)
rm(raw); invisible(gc())
obj <- PercentageFeatureSet(obj, pattern = "^MT-", assay = "RAW", col.name = "percent.mt")
obj$n_before_qc <- 1L
cat("MT genes found:", sum(grepl("^MT-", rownames(obj))), "\n")
obj

*What you should see:* An object summary with two assays (RNA and RAW) and a count of mitochondrial genes (about 13).

## 4. Ambient RNA, doublets and the mitochondrial cut-off
Order: **ambient RNA -> doublets -> mitochondrial % (15%)**. Every step is logged per library and per group so that QC itself
cannot silently create a difference (guide, step 8). The mitochondrial % is measured on the **original** counts (`RAW`).

**Cell annotation:** **Ambient RNA.** Tabula Sapiens already ran DecontX per sample and stored it as a layer; if it is there it is used (`ambient = "auto"`). Otherwise DecontX is run here, library by library. Either way, the original counts stay in `RAW`.

In [ ]:
amb_used <- "none"
dx_file <- file.path(export_dir, "decontX_counts.mtx.gz")
if (ambient == "none") {
  amb_used <- "none (original counts)"
} else if (ambient == "auto" && file.exists(dx_file)) {
  dx <- as(Matrix::readMM(dx_file), "CsparseMatrix"); rownames(dx) <- rownames(obj); colnames(dx) <- colnames(obj)
  dx@x <- round(dx@x)                                  # edgeR/Seurat want whole counts
  obj[["RNA"]] <- CreateAssay5Object(counts = dx); rm(dx); amb_used <- "provided decontXcounts layer (rounded)"
} else if (have("celda")) {
  cnt <- LayerData(obj, assay = "RAW", layer = "counts"); out <- cnt
  for (lib in unique(obj$library)) {                   # DecontX is run per capture library
    ix <- which(obj$library == lib); if (length(ix) < 50) next
    res <- celda::decontX(cnt[, ix], verbose = FALSE)
    out[, ix] <- round(res$decontXcounts)
  }
  obj[["RNA"]] <- CreateAssay5Object(counts = out); rm(cnt, out); amb_used <- "DecontX run here, per library"
} else {
  warning("celda is not installed and no decontX layer was found: ambient RNA was NOT corrected. install.packages('BiocManager'); BiocManager::install('celda')")
  amb_used <- "NOT corrected (celda missing)"
}
DefaultAssay(obj) <- "RNA"
cat("Ambient RNA:", amb_used, "\n")
lost <- 1 - Matrix::colSums(LayerData(obj, assay = "RNA", layer = "counts")) / pmax(Matrix::colSums(LayerData(obj, assay = "RAW", layer = "counts")), 1)
obj$ambient_removed_frac <- lost
print(summary(obj$ambient_removed_frac))

*What you should see:* The method used and a summary of the fraction of counts removed per cell (often a few percent; much larger values in some cells suggest heavy contamination).

**Cell annotation:** **Doublets** with scDblFinder, run separately for each capture library. Plate-based smart-seq2 cells are skipped because one well holds one cell.

In [ ]:
obj$doublet_class <- "not_tested"; obj$doublet_score <- NA_real_
if (run_doublets && have("scDblFinder") && have("SingleCellExperiment")) {
  for (lib in unique(obj$library)) {
    ix <- which(obj$library == lib & !grepl("smart", obj$method, ignore.case = TRUE))
    if (length(ix) < 200) next
    sce <- SingleCellExperiment::SingleCellExperiment(assays = list(counts = LayerData(obj, assay = "RAW", layer = "counts")[, ix]))
    sce <- scDblFinder::scDblFinder(sce, verbose = FALSE)
    obj$doublet_class[ix] <- as.character(sce$scDblFinder.class); obj$doublet_score[ix] <- sce$scDblFinder.score
  }
} else message("Doublet detection skipped (turned off or scDblFinder missing): BiocManager::install('scDblFinder')")
print(table(obj$library, obj$doublet_class))

*What you should see:* Per library: how many cells are singlets, doublets or not tested. About 2-8% doublets is typical for 10x.

**Cell annotation:** **Mitochondrial cut-off and the exclusion log.** First the per-library distributions (so you can see whether 15% is sensible), then the filter. The log shows what is lost per library and per group.

In [ ]:
qc_lib <- obj@meta.data %>% group_by(library, method) %>%
  summarise(cells = n(), median_UMI = median(nCount_RNA), median_genes = median(nFeature_RNA),
            median_mito = round(median(percent.mt), 2), p95_mito = round(quantile(percent.mt, .95), 2),
            above_mito_cut = sum(percent.mt > max_mito_pct), doublets = sum(doublet_class == "doublet"), .groups = "drop")
print(as.data.frame(qc_lib)); write.csv(qc_lib, file.path(out_dir, "qc_per_library_before.csv"), row.names = FALSE)

obj$fail_mito    <- obj$percent.mt > max_mito_pct
obj$fail_low     <- obj$nFeature_RNA < min_genes | obj$nCount_RNA < min_counts
obj$fail_doublet <- obj$doublet_class == "doublet"
obj$keep <- !(obj$fail_mito | obj$fail_low | obj$fail_doublet)
log_tab <- obj@meta.data %>% group_by(library, group, depot) %>%
  summarise(before = n(), mito = sum(fail_mito), low = sum(fail_low), doublet = sum(fail_doublet), kept = sum(keep),
            pct_kept = round(100 * mean(keep), 1), .groups = "drop")
print(as.data.frame(log_tab)); write.csv(log_tab, file.path(out_dir, "exclusion_log.csv"), row.names = FALSE)
cat("\nRetention by group (QC should not differ much between groups):\n")
print(obj@meta.data %>% group_by(group) %>% summarise(cells = n(), pct_kept = round(100 * mean(keep), 1)))

p <- ggplot(obj@meta.data, aes(percent.mt)) + geom_histogram(bins = 80) + geom_vline(xintercept = max_mito_pct, colour = "red") +
  facet_wrap(~library, scales = "free_y") + labs(title = paste0("Mitochondrial % per library (red = ", max_mito_pct, "% cut-off)"))
ggsave(file.path(out_dir, "qc_mito_histograms.png"), p, width = 11, height = 7, dpi = 130)
print(p)
obj <- subset(obj, subset = keep)
cat("cells kept:", ncol(obj), "\n")

*What you should see:* A per-library QC table, the exclusion log, retention by group, and a histogram per library with the red 15% line. If one library loses most of its cells, stop and look at it.

## 5. Normalise, elbow plot, clusters and UMAP (all cells)
Two representations are kept, as your guide says: **log-normalised** values and PCA for clustering and plots, and
**un-normalised counts** for the statistics. Clustering uses the ambient-corrected counts; the PCA is integrated across donors
with Seurat (RPCA) so that donors mix. Disease is never regressed out.

**Cell annotation:** A reusable function for the whole reduction. It is used here and again for the endothelial cells.

In [ ]:
sig_pcs <- function(stdev) {            # the published atlas's rule for the number of significant PCs
  pct <- stdev / sum(stdev) * 100; cum <- cumsum(pct)
  co1 <- which(cum > 90 & pct < 5)[1]
  co2 <- sort(which((pct[-length(pct)] - pct[-1]) > 0.05), decreasing = TRUE)[1] + 1
  min(co1, co2, na.rm = TRUE)
}
make_batches <- function(x, min_n) { x <- as.character(x); tb <- table(x); small <- names(tb)[tb < min_n]; x[x %in% small] <- "pooled_small"; x }

reduce_obj <- function(obj, batch_col = "donor", resolution = 0.3, tag = "all") {
  DefaultAssay(obj) <- "RNA"
  obj$integ_batch <- make_batches(obj[[batch_col]][, 1], min_cells_batch)
  use_int <- integrate && length(unique(obj$integ_batch)) > 1
  if (use_int) obj[["RNA"]] <- split(obj[["RNA"]], f = obj$integ_batch)
  obj <- NormalizeData(obj, normalization.method = "LogNormalize", scale.factor = 10000, verbose = FALSE)
  obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = n_var_genes, verbose = FALSE)
  obj <- ScaleData(obj, vars.to.regress = "nCount_RNA", verbose = FALSE)
  obj <- RunPCA(obj, features = VariableFeatures(obj), npcs = 50, verbose = FALSE)
  red <- "pca"
  if (use_int) {
    kw <- 100
    ok <- FALSE
    while (!ok && kw >= 5) {
      res <- tryCatch(IntegrateLayers(obj, method = RPCAIntegration, orig.reduction = "pca", new.reduction = "integrated",
                                      k.weight = kw, verbose = FALSE), error = function(e) NULL)
      if (is.null(res)) kw <- floor(kw / 2) else { obj <- res; ok <- TRUE }
    }
    if (ok) red <- "integrated" else warning("Integration failed for '", tag, "'; plain PCA was used instead.")
    obj <- JoinLayers(obj)
  }
  sd_all <- Stdev(obj, reduction = "pca")
  n_sig <- sig_pcs(sd_all); n_use <- min(n_sig, max_pcs, 50)
  p_el <- ElbowPlot(obj, ndims = 50, reduction = "pca") + geom_vline(xintercept = n_use, colour = "red", linetype = 2) +
    labs(title = paste0("Elbow plot (", tag, "): rule picks ", n_sig, " PCs, ", n_use, " used"))
  ggsave(file.path(out_dir, paste0("elbow_", tag, ".png")), p_el, width = 6, height = 4, dpi = 130); print(p_el)
  obj <- FindNeighbors(obj, reduction = red, dims = 1:n_use, verbose = FALSE)
  obj <- FindClusters(obj, resolution = resolution, random.seed = 1, verbose = FALSE)
  obj <- RunUMAP(obj, reduction = red, dims = 1:n_use, seed.use = 1, verbose = FALSE)
  obj$cluster <- as.character(obj$seurat_clusters)
  attr(obj, "n_pcs") <- n_use; attr(obj, "reduction_used") <- red
  cat("[", tag, "] PCs used:", n_use, "| reduction:", red, "| clusters:", length(unique(obj$cluster)), "\n")
  obj
}

**Cell annotation:** Run it on all cells: normalise, variable genes, PCA, **elbow plot**, integrate, cluster, UMAP.

In [ ]:
obj <- reduce_obj(obj, batch_col = "donor", resolution = res_all, tag = "all_cells")
p1 <- DimPlot(obj, group.by = "cluster", label = TRUE, raster = FALSE) + ggtitle("All cells: clusters")
p2 <- DimPlot(obj, group.by = "donor", raster = FALSE) + ggtitle("All cells: donor (should be mixed)")
p3 <- DimPlot(obj, group.by = "depot", raster = FALSE) + ggtitle("All cells: depot")
for (nm in c("p1","p2","p3")) { print(get(nm)); ggsave(file.path(out_dir, paste0("umap_all_", nm, ".png")), get(nm), width = 7, height = 5.5, dpi = 130) }
print(table(obj$cluster))

*What you should see:* The elbow plot with a red line at the chosen number of PCs, then three UMAPs. Donors should overlap within clusters; if a cluster is one donor only, it may be technical.

## 6. Annotate the clusters: DEGs plus Human Protein Atlas markers
For each cluster the notebook (a) finds its **differentially expressed genes** (DEGs, one cluster against all others),
(b) checks which of them are **Human Protein Atlas (HPA)** marker genes for each cell type, and (c) scores the cluster on
each marker panel (mean z-score across clusters). A cluster gets a name only if its best score is clear *and* ahead of the
runner-up by a margin; otherwise it stays `Unassigned`.

*Be careful with two things.*
1. **The marker lists below were written from my memory of the HPA single-cell-type data, not downloaded.** I could not
   check them against proteinatlas.org from my workspace. Treat them as a starting panel, check them on the HPA site, and
   replace them with your own export through `hpa_markers_csv` if you prefer.
2. **PECAM1 and CDH5 are not used to name clusters**, although your guide lists them as endothelial identity genes. They are
   two of the genes you are testing, so using them to pick the cells would bias their own result. They are still drawn on
   the dot plots. Endothelial identity is scored with VWF, KDR, CLDN5, EGFL7, RAMP2 and others.

**Cell annotation:** The marker panels (broad cell types, from the HPA / your guide) and the endothelial subtype panels (from your guide). The subtype panels avoid the tested genes and avoid FABP4/CD36, which the guide warns are real capillary endothelial genes.

In [ ]:
broad_panels <- list(
  Endothelial       = c("VWF","KDR","CLDN5","EGFL7","RAMP2","EMCN","FLT1","ECSCR"),
  Lymphatic_EC      = c("PROX1","MMRN1","FLT4","PDPN","LYVE1","CCL21"),
  Pericyte          = c("PDGFRB","RGS5","CSPG4","KCNJ8","NOTCH3"),
  Smooth_muscle     = c("MYH11","ACTA2","TAGLN","CNN1","DES"),
  Fibroblast_ASPC   = c("PDGFRA","COL1A1","DCN","LUM","CFD","APOD"),
  Adipocyte         = c("ADIPOQ","PLIN1","LEP","CIDEA","LPL"),
  Macrophage        = c("CD68","CD163","MRC1","CSF1R","C1QA","MS4A7"),
  T_cell            = c("CD3E","CD3D","CD2","IL7R","TRAC"),
  NK_cell           = c("NKG7","GNLY","KLRD1","PRF1"),
  B_cell            = c("MS4A1","CD79A","CD79B","CD19"),
  Mast_cell         = c("KIT","TPSAB1","TPSB2","CPA3"),
  Neutrophil        = c("S100A8","S100A9","FCGR3B","CSF3R"),
  Mesothelial       = c("MSLN","UPK3B","KRT19","WT1")
)
ec_panels <- list(
  Capillary_EC = c("RGCC","CA4","RBP7","GPIHBP1","CD36","FABP4"),    # CD36/FABP4 only as support, see note
  Arterial_EC  = c("GJA5","NEBL","SEMA3G","HEY1","FBLN5","SOX17"),
  Venous_EC    = c("ACKR1","NR2F2","SELP","PLVAP","VWF"),
  Lymphatic_EC = c("PROX1","MMRN1","FLT4","PDPN","LYVE1","CCL21")
)
ec_panels$Capillary_EC <- setdiff(ec_panels$Capillary_EC, c("CD36","FABP4"))     # kept out of naming on purpose
if (nzchar(hpa_markers_csv) && file.exists(hpa_markers_csv)) {
  h <- read.csv(hpa_markers_csv, stringsAsFactors = FALSE); broad_panels <- split(h$gene, h$cell_type)
  cat("Using your HPA table:", length(broad_panels), "cell types\n")
}
excluded_from_naming <- panel_genes
broad_panels <- lapply(broad_panels, setdiff, excluded_from_naming)
ec_panels    <- lapply(ec_panels,    setdiff, excluded_from_naming)
tab <- data.frame(panel = rep(names(broad_panels), lengths(broad_panels)), gene = unlist(broad_panels))
tab$in_data <- tab$gene %in% rownames(obj); print(table(tab$panel, tab$in_data))
write.csv(tab, file.path(out_dir, "marker_panels_used.csv"), row.names = FALSE)

*What you should see:* A table of how many panel genes exist in your data for each cell type (FALSE counts are genes missing from the file, which is fine in small numbers).

**Cell annotation:** Two functions: the cluster DEGs, and the panel scoring that turns them into names.

In [ ]:
cluster_degs <- function(obj, group = "cluster", top_n = 30) {
  Idents(obj) <- group
  m <- FindAllMarkers(obj, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.5, test.use = "wilcox", verbose = FALSE)
  if (!nrow(m)) return(m)
  m %>% group_by(cluster) %>% arrange(desc(avg_log2FC), .by_group = TRUE) %>% slice_head(n = top_n) %>% ungroup()
}
score_clusters <- function(obj, panels, degs, group = "cluster", prefix = "") {
  genes <- intersect(unique(unlist(panels)), rownames(obj))
  avg <- AverageExpression(obj, assays = "RNA", features = genes, group.by = group, layer = "data", verbose = FALSE)$RNA
  avg <- log1p(as.matrix(avg)); colnames(avg) <- sub("^g", "", colnames(avg))
  z <- t(scale(t(avg))); z[is.na(z)] <- 0
  sc <- sapply(panels, function(g) { g <- intersect(g, rownames(z)); if (length(g)) colMeans(z[g, , drop = FALSE]) else rep(NA_real_, ncol(z)) })
  sc <- as.matrix(sc); rownames(sc) <- colnames(z)
  out <- lapply(rownames(sc), function(cl) {
    s <- sort(sc[cl, ], decreasing = TRUE, na.last = NA); top <- names(s)[1]; margin <- if (length(s) > 1) s[1] - s[2] else s[1]
    d <- degs$gene[degs$cluster == cl]
    hits <- intersect(d, panels[[top]])
    data.frame(cluster = cl, best_panel = top, score = round(s[1], 2), runner_up = if (length(s) > 1) names(s)[2] else NA,
               margin = round(margin, 2), n_DEG_in_panel = length(hits), DEG_markers = paste(hits, collapse = ","),
               top5_DEGs = paste(head(d, 5), collapse = ","),
               label = if (s[1] >= min_score && margin >= min_margin) top else "Unassigned", stringsAsFactors = FALSE)
  })
  do.call(rbind, out)
}

**Cell annotation:** Run the DEGs and the scoring for all cells, then write the names into the object. **Read the table:** a label with `n_DEG_in_panel` of 0 or a small margin deserves a second look.

In [ ]:
degs_all <- cluster_degs(obj)
write.csv(degs_all, file.path(out_dir, "cluster_DEGs_all_cells.csv"), row.names = FALSE)
ann_all <- score_clusters(obj, broad_panels, degs_all)
print(ann_all[, c("cluster","label","score","margin","runner_up","n_DEG_in_panel","DEG_markers","top5_DEGs")], row.names = FALSE)
write.csv(ann_all, file.path(out_dir, "cluster_annotation_all_cells.csv"), row.names = FALSE)
obj$cell_type <- ann_all$label[match(obj$cluster, ann_all$cluster)]
cat("\nCells per assigned type:\n"); print(sort(table(obj$cell_type), decreasing = TRUE))
# cross-check against the label Tabula Sapiens itself provides (NOT used to name anything)
if (!all(is.na(obj$TS_label))) {
  xt <- as.data.frame.matrix(table(obj$cell_type, obj$TS_label)); write.csv(xt, file.path(out_dir, "check_vs_TabulaSapiens_labels.csv"))
  cat("\nFor each of our types, the commonest Tabula Sapiens label:\n")
  print(sapply(split(obj$TS_label, obj$cell_type), function(x) names(sort(table(x), decreasing = TRUE))[1]))
}

*What you should see:* The annotation table (one row per cluster), the cell counts per type, and a cross-check with Tabula Sapiens' own labels. The cross-check is a sanity test only.

**Cell annotation:** Plots of the annotation: UMAP by type and a dot plot of the markers (the two tested identity genes PECAM1 and CDH5 are shown here for information).

In [ ]:
p <- DimPlot(obj, group.by = "cell_type", label = TRUE, repel = TRUE, raster = FALSE) + ggtitle("All cells: assigned cell type")
print(p); ggsave(file.path(out_dir, "umap_all_cell_type.png"), p, width = 8, height = 6, dpi = 130)
show <- intersect(unique(c(unlist(lapply(broad_panels, head, 3)), "PECAM1", "CDH5", "ERG")), rownames(obj))
Idents(obj) <- "cell_type"
d <- DotPlot(obj, features = show) + RotatedAxis() + ggtitle("Marker dot plot by assigned type")
print(d); ggsave(file.path(out_dir, "dotplot_all_cell_type.png"), d, width = 12, height = 5, dpi = 130)

*What you should see:* A labelled UMAP and a dot plot. Each type should light up for its own markers.

## 7. The endothelial population as its own object
All clusters named `Endothelial` or `Lymphatic_EC` are put into a **second object**. The pipeline is repeated on them
(variable genes, PCA, **elbow plot**, clusters, **UMAP**). ERG positivity is **not** used to choose these cells (guide, step 10).
Clusters that look endothelial but are not clear stay `Unassigned` and are listed, not hidden.

**Cell annotation:** Subset the endothelial cells and re-run the reduction on them.

In [ ]:
ec_types <- c("Endothelial", "Lymphatic_EC")
ec_cells <- colnames(obj)[obj$cell_type %in% ec_types]
cat("endothelial cells selected:", length(ec_cells), "of", ncol(obj), "\n")
unassigned_cl <- ann_all$cluster[ann_all$label == "Unassigned"]
cat("Unassigned clusters (not in the EC object; check if any look endothelial):", paste(unassigned_cl, collapse = ", "), "\n")
if (length(ec_cells) < 100) stop("Fewer than 100 endothelial cells: check the annotation table in section 6 before going on.")
ec <- subset(obj, cells = ec_cells)
for (a in setdiff(Assays(ec), "RNA")) if (a != "RAW") ec[[a]] <- NULL
ec <- reduce_obj(ec, batch_col = "donor", resolution = res_ec, tag = "endothelial")
p <- DimPlot(ec, group.by = "cluster", label = TRUE, raster = FALSE) + ggtitle("Endothelial cells: clusters")
print(p); ggsave(file.path(out_dir, "umap_ec_clusters.png"), p, width = 7, height = 5.5, dpi = 130)

*What you should see:* A second elbow plot (for the EC subset) and the endothelial UMAP with clusters.

**Cell annotation:** **Annotate the endothelial subtypes** with the guide's panels (capillary, arterial, venous, lymphatic). Mixed-lineage or unclear clusters stay `Unassigned_EC` for review. The identity panel is also scored so that non-endothelial contamination shows up.

In [ ]:
all_panels <- c(ec_panels, list(Pericyte = broad_panels$Pericyte, Smooth_muscle = broad_panels$Smooth_muscle,
                                Immune = unique(unlist(broad_panels[c("Macrophage","T_cell","NK_cell","B_cell","Mast_cell","Neutrophil")])),
                                Fibroblast_ASPC = broad_panels$Fibroblast_ASPC, Adipocyte = broad_panels$Adipocyte))
degs_ec <- cluster_degs(ec)
write.csv(degs_ec, file.path(out_dir, "cluster_DEGs_endothelial.csv"), row.names = FALSE)
ann_ec <- score_clusters(ec, all_panels, degs_ec)
ann_ec$label <- ifelse(ann_ec$label %in% names(ec_panels), ann_ec$label,
                       ifelse(ann_ec$label == "Unassigned", "Unassigned_EC", paste0("Non-EC_", ann_ec$label)))
print(ann_ec[, c("cluster","label","score","margin","runner_up","n_DEG_in_panel","DEG_markers","top5_DEGs")], row.names = FALSE)
write.csv(ann_ec, file.path(out_dir, "cluster_annotation_endothelial.csv"), row.names = FALSE)
ec$ec_subtype <- ann_ec$label[match(ec$cluster, ann_ec$cluster)]
ec$ec_class   <- ifelse(ec$ec_subtype == "Lymphatic_EC", "Lymphatic", ifelse(grepl("^Non-EC", ec$ec_subtype), "Non-EC", "Blood_EC"))
print(table(ec$ec_subtype, ec$ec_class))
p <- DimPlot(ec, group.by = "ec_subtype", label = TRUE, repel = TRUE, raster = FALSE) + ggtitle("Endothelial subtypes")
print(p); ggsave(file.path(out_dir, "umap_ec_subtype.png"), p, width = 7.5, height = 5.5, dpi = 130)
Idents(ec) <- "ec_subtype"
feat <- intersect(unique(c(unlist(lapply(ec_panels, head, 3)), "PECAM1", "CDH5", "ERG")), rownames(ec))
print(DotPlot(ec, features = feat) + RotatedAxis() + ggtitle("Endothelial subtype markers"))

*What you should see:* The subtype table, a UMAP of subtypes and a dot plot. `Non-EC_...` clusters are contaminants kept out of the endothelial tests; `Unassigned_EC` ones need your eye.

## 8. Separate SAT and VAT, and look at ERG (no statistical claim yet)
Your guide (steps 11-12): count cells for every **donor x depot x subtype**, check that subtypes are supported by several
donors, then look at ERG before testing it. A zero count means *not detected*, not *absent*.

**Cell annotation:** Coverage table (cells per donor, depot and subtype) and the check for the depot split.

In [ ]:
ec_blood <- subset(ec, subset = ec_class %in% c("Blood_EC", "Lymphatic"))
cov <- ec_blood@meta.data %>% count(donor, depot, group, ec_subtype, name = "cells") %>% arrange(depot, ec_subtype, donor)
write.csv(cov, file.path(out_dir, "coverage_donor_depot_subtype.csv"), row.names = FALSE)
print(as.data.frame(cov))
cat("\nEndothelial cells by depot:\n"); print(table(ec_blood$depot, useNA = "ifany"))
cat("\nDonors with >=", min_cells_sample, "ECs, by depot and group:\n")
print(ec_blood@meta.data %>% count(donor, depot, group) %>% filter(n >= min_cells_sample) %>% count(depot, group, name = "donors"))
if (!has_depot) message("SAT and VAT are not both present in the metadata. Check the depot mapping in section 3 (sat_regex / vat_regex).")

*What you should see:* A long table, then the cell counts per depot and donors with enough ECs. Few donors per cell means the later tests will have little power.

**Cell annotation:** **ERG expression in SAT and in VAT**: UMAP coloured by ERG (same colour scale), a dot plot by subtype and depot, the fraction of cells with ERG detected per donor, and ERG detection against sequencing depth.

In [ ]:
if ("ERG" %in% rownames(ec_blood)) {
  ec_blood$sub_depot <- paste(ec_blood$ec_subtype, ec_blood$depot, sep = " | ")
  lim <- c(0, max(1, quantile(LayerData(ec_blood, assay = "RNA", layer = "data")["ERG", ], 0.99)))
  p <- FeaturePlot(ec_blood, "ERG", split.by = "depot", raster = FALSE, keep.scale = "all") & scale_colour_gradient(low = "grey85", high = "firebrick", limits = lim)
  print(p); ggsave(file.path(out_dir, "ERG_featureplot_by_depot.png"), p, width = 11, height = 5, dpi = 130)
  Idents(ec_blood) <- "sub_depot"
  print(DotPlot(ec_blood, features = "ERG") + RotatedAxis() + ggtitle("ERG by subtype and depot"))
  cnt <- LayerData(ec_blood, assay = "RNA", layer = "counts")
  ec_blood$ERG_counts <- as.numeric(cnt["ERG", ]); ec_blood$ERG_detected <- ec_blood$ERG_counts > 0
  per_donor <- ec_blood@meta.data %>% group_by(donor, depot, group) %>%
    summarise(ECs = n(), ERG_reads = sum(ERG_counts), pct_cells_ERG = round(100 * mean(ERG_detected), 1),
              median_UMI = median(nCount_RNA), .groups = "drop")
  print(as.data.frame(per_donor)); write.csv(per_donor, file.path(out_dir, "ERG_per_donor_depot.csv"), row.names = FALSE)
  print(ggplot(ec_blood@meta.data, aes(log10(nCount_RNA), as.numeric(ERG_detected))) + geom_smooth(method = "loess", formula = y ~ x) +
          labs(y = "fraction with ERG detected", title = "ERG detection rises with depth: compare like with like"))
} else message("ERG is not in the object.")

*What you should see:* ERG maps by depot, a dot plot, and a per-donor table of ERG reads. If ERG reads are very few, the test later will rightly say 'insufficient information'.

## 9. Pseudobulk: ERG and the panel, control vs T2D (when possible)
**Donors are the replicates.** Raw counts are summed across cells within **donor x depot** (all blood ECs), and again within
**donor x depot x subtype** for the subtypes that enough donors support. Samples with fewer than
20 ECs are left out as *missing* (not zero).

| Technique | Role here |
|---|---|
| Log-normalisation per cell | clustering, UMAPs, annotation checks only |
| **Pseudobulk + edgeR TMM** | **primary** test (quasi-likelihood F-test, robust) |
| Pseudobulk + DESeq2 (median-of-ratios) | alternative, run if the package is installed |
| Covariates (sex, age, method) or matched donors | added **only** if the design has enough donors; the notebook checks model rank and residual degrees of freedom first |

*Sign convention:* log2FC is **T2D relative to Control** (or **VAT relative to SAT** for the depot comparison). FDR < 0.05 is
the threshold, applied across the panel genes tested within one analysis. Sample sizes are unequal when the two groups
differ in donors; TMM and median-of-ratios address library composition, **not** study design, so differences in age, sex or
method between groups are handled by the covariate step, and any leftover confounding is stated in the results.

**Cell annotation:** Helper functions: build pseudobulk counts, choose covariates that the design can afford, run edgeR (and DESeq2 if available).

In [ ]:
pseudobulk <- function(o, by = c("donor", "depot")) {
  md <- o@meta.data; key <- do.call(paste, c(md[by], sep = "||"))
  cnt <- LayerData(o, assay = "RNA", layer = "counts")
  f <- factor(key); M <- sparseMatrix(i = seq_along(f), j = as.integer(f), x = 1, dims = c(length(f), nlevels(f)), dimnames = list(NULL, levels(f)))
  pb <- as.matrix(cnt %*% M)
  meta <- md[match(levels(f), key), unique(c(by, "group", "sex", "age_num", "method")), drop = FALSE]
  meta$n_cells <- as.integer(table(f)[levels(f)]); meta$sample <- levels(f); rownames(meta) <- meta$sample
  list(counts = pb, meta = meta)
}
afford_covariates <- function(meta, main, cands = c("sex", "age_num"), min_resid_df = 2) {
  keep <- character(0)
  for (cv in cands) {
    if (!cv %in% colnames(meta) || all(is.na(meta[[cv]])) || length(unique(na.omit(meta[[cv]]))) < 2) next
    d <- model.matrix(as.formula(paste("~", paste(c(main, keep, cv), collapse = " + "))), data = meta)
    if (qr(d)$rank == ncol(d) && nrow(d) - ncol(d) >= min_resid_df) keep <- c(keep, cv)
  }
  keep
}
run_edger <- function(pb, main, level_ref, level_test, extra = character(0), block = NULL, force = panel_genes) {
  meta <- pb$meta; meta[[main]] <- factor(meta[[main]], levels = c(level_ref, level_test))
  terms <- c(extra, main); if (!is.null(block)) terms <- c(block, terms)
  ok <- complete.cases(meta[, c(terms)]); meta <- meta[ok, ]; counts <- pb$counts[, rownames(meta)]
  design <- model.matrix(as.formula(paste("~", paste(terms, collapse = " + "))), data = meta)
  if (qr(design)$rank < ncol(design)) return(list(status = "design not full rank", tab = NULL))
  if (nrow(design) - ncol(design) < 1) return(list(status = "no residual degrees of freedom", tab = NULL))
  y <- DGEList(counts = counts)
  keep <- filterByExpr(y, design = design) | rownames(y) %in% force
  y <- y[keep, , keep.lib.sizes = FALSE]
  y <- tryCatch(normLibSizes(y), error = function(e) calcNormFactors(y, method = "TMM"))
  y <- estimateDisp(y, design, robust = TRUE)
  fit <- glmQLFit(y, design, robust = TRUE)
  coefn <- grep(paste0("^", main), colnames(design))
  tt <- topTags(glmQLFTest(fit, coef = coefn), n = Inf, adjust.method = "BH")$table
  tt$gene <- rownames(tt); tt$reads_total <- rowSums(counts)[tt$gene]
  tt$n_ref <- sum(meta[[main]] == level_ref); tt$n_test <- sum(meta[[main]] == level_test)
  tt$covariates <- paste(extra, collapse = "+"); tt$block <- ifelse(is.null(block), "", block)
  list(status = "ok", tab = tt, y = y, meta = meta)
}
run_deseq2 <- function(pb, main, level_ref, level_test, extra = character(0), block = NULL) {
  if (!have("DESeq2")) return(NULL)
  meta <- pb$meta; meta[[main]] <- factor(meta[[main]], levels = c(level_ref, level_test))
  terms <- c(block, extra, main); frm <- as.formula(paste("~", paste(terms, collapse = " + ")))
  dds <- DESeq2::DESeqDataSetFromMatrix(round(pb$counts[, rownames(meta)]), meta, frm)
  dds <- dds[rowSums(DESeq2::counts(dds) >= 10) >= 2 | rownames(dds) %in% panel_genes, ]
  dds <- tryCatch(DESeq2::DESeq(dds, quiet = TRUE), error = function(e) NULL)
  if (is.null(dds)) return(NULL)
  r <- as.data.frame(DESeq2::results(dds, name = grep(paste0("^", main), DESeq2::resultsNames(dds), value = TRUE)[1]))
  r$gene <- rownames(r); r
}
panel_table <- function(res, label) {
  if (is.null(res$tab)) return(data.frame(analysis = label, note = res$status))
  t <- res$tab[res$tab$gene %in% panel_genes, ]
  if (!nrow(t)) return(data.frame(analysis = label, note = "panel genes filtered out"))
  data.frame(analysis = label, gene = t$gene, log2FC = round(t$logFC, 3), PValue = signif(t$PValue, 3), FDR_in_analysis = NA_real_,
             reads_total = t$reads_total, n_ref = t$n_ref, n_test = t$n_test, covariates = t$covariates)
}

**Cell annotation:** **The primary analysis.** Build the pseudobulk per donor x depot, then run **control vs T2D within each depot** if the audit allowed it. If there is no T2D group, run **SAT vs VAT** (paired by donor) instead. Both are labelled in the output.

In [ ]:
ecb <- subset(ec_blood, subset = ec_class == "Blood_EC")                     # blood endothelial cells (lymphatics separate)
pb_all <- pseudobulk(ecb, by = c("donor", "depot"))
keep_s <- pb_all$meta$n_cells >= min_cells_sample & pb_all$meta$depot %in% c("SAT", "VAT")
cat("samples kept (>=", min_cells_sample, "ECs):", sum(keep_s), "of", length(keep_s), "\n")
pb_all$meta <- pb_all$meta[keep_s, ]; pb_all$counts <- pb_all$counts[, keep_s, drop = FALSE]
write.csv(pb_all$meta, file.path(out_dir, "pseudobulk_samples.csv"), row.names = FALSE)
print(pb_all$meta[, c("donor","depot","group","sex","n_cells")], row.names = FALSE)

results <- list(); mode_used <- "none"
if (has_t2d) {
  mode_used <- "T2D_vs_Control"
  for (dp in c("SAT", "VAT")) {
    sub <- list(counts = pb_all$counts[, pb_all$meta$depot == dp, drop = FALSE], meta = pb_all$meta[pb_all$meta$depot == dp, ])
    sub$meta <- sub$meta[sub$meta$group %in% c("Control", "T2D"), ]; sub$counts <- sub$counts[, rownames(sub$meta), drop = FALSE]
    nC <- sum(sub$meta$group == "Control"); nT <- sum(sub$meta$group == "T2D")
    if (nC < 2 || nT < 2) { cat(dp, ": Control", nC, "T2D", nT, "-> insufficient donors, no test\n"); next }
    cv <- afford_covariates(sub$meta, "group")
    cat(dp, ": Control", nC, "| T2D", nT, "| covariates used:", ifelse(length(cv), paste(cv, collapse = ", "), "none"), "\n")
    r <- run_edger(sub, "group", "Control", "T2D", extra = cv)
    results[[dp]] <- r; r$dds <- run_deseq2(sub, "group", "Control", "T2D", extra = cv)
    results[[dp]]$deseq2 <- r$dds
  }
} else if (has_depot) {
  mode_used <- "SAT_vs_VAT_paired"
  both <- names(which(table(pb_all$meta$donor[pb_all$meta$depot %in% c("SAT","VAT")]) == 2))
  sub <- list(counts = pb_all$counts[, pb_all$meta$donor %in% both, drop = FALSE], meta = pb_all$meta[pb_all$meta$donor %in% both, ])
  cat("paired donors (both depots with enough ECs):", length(both), "\n")
  if (length(both) >= 3) {
    r <- run_edger(sub, "depot", "SAT", "VAT", block = "donor")
    results[["SATvsVAT"]] <- r; results[["SATvsVAT"]]$deseq2 <- run_deseq2(sub, "depot", "SAT", "VAT", block = "donor")
  } else cat("fewer than 3 paired donors: insufficient information, no test is run.\n")
}
cat("\nAnalysis performed:", mode_used, "\n")

*What you should see:* The kept pseudobulk samples, then either control-vs-T2D per depot or SAT-vs-VAT, with donor numbers and the covariates the design could afford. 'insufficient donors' is a valid, honest result.

**Cell annotation:** Collect the results: the **ERG line** first, then the whole panel. The FDR is recomputed across the panel genes of each analysis, and the full gene-wide table is also saved with the edgeR FDR.

In [ ]:
panel_out <- list()
for (nm in names(results)) {
  r <- results[[nm]]
  if (is.null(r$tab)) { cat(nm, ":", r$status, "\n"); next }
  full <- r$tab; write.csv(full, file.path(out_dir, paste0("DE_genewide_", mode_used, "_", nm, "_edgeR.csv")), row.names = FALSE)
  pt <- panel_table(r, nm); pt$FDR_in_analysis <- signif(p.adjust(pt$PValue, "BH"), 3)
  pt$FDR_genewide <- signif(full$FDR[match(pt$gene, full$gene)], 3)
  if (!is.null(r$deseq2)) { d <- r$deseq2; pt$DESeq2_log2FC <- round(d$log2FoldChange[match(pt$gene, d$gene)], 3); pt$DESeq2_p <- signif(d$pvalue[match(pt$gene, d$gene)], 3) }
  pt$call <- ifelse(pt$FDR_in_analysis < fdr_cut, "significant (FDR<0.05)", "not significant")
  pt$caution <- ifelse(pt$reads_total < 50, "very few reads: insufficient information", ifelse(pmin(pt$n_ref, pt$n_test) < 3, "fewer than 3 donors in a group", ""))
  panel_out[[nm]] <- pt
}
panel_res <- if (length(panel_out)) bind_rows(panel_out) else data.frame(note = "no test was run (see messages above)")
print(panel_res, row.names = FALSE); write.csv(panel_res, file.path(out_dir, "panel_results.csv"), row.names = FALSE)
cat("\nERG:\n"); if ("gene" %in% names(panel_res)) print(panel_res[panel_res$gene == "ERG", ], row.names = FALSE)

*What you should see:* A table with log2FC, p, FDR, reads, donor numbers and a cautionary note for every panel gene. ERG is reprinted on its own.

**Cell annotation:** **Check for outliers and confounding** (guide, step 13): an MDS of the pseudobulk samples coloured by depot and group. Samples that sit apart for technical reasons (method, donor) are a warning.

In [ ]:
if (ncol(pb_all$counts) >= 4) {
  y <- DGEList(pb_all$counts); y <- y[filterByExpr(y), , keep.lib.sizes = FALSE]
  y <- tryCatch(normLibSizes(y), error = function(e) calcNormFactors(y))
  mds <- plotMDS(y, plot = FALSE)
  df <- data.frame(x = mds$x, y = mds$y, pb_all$meta[colnames(y), c("donor","depot","group","method")])
  p <- ggplot(df, aes(x, y, colour = depot, shape = group, label = donor)) + geom_point(size = 3) + geom_text(vjust = -1, size = 3, show.legend = FALSE) +
    labs(title = "Pseudobulk MDS (TMM-normalised)", x = "dim 1", y = "dim 2")
  print(p); ggsave(file.path(out_dir, "pseudobulk_MDS.png"), p, width = 7, height = 5, dpi = 130)
}

*What you should see:* One dot per donor x depot. Dots of the same depot clustering together is expected; a single donor far from all others is an outlier to investigate.

## 10. Bar graph: ERG, CD31, VE-cadherin, eNOS, VCAM1, ICAM1, HIF1-alpha, ANGPT2
Each bar is the **group mean of TMM-normalised log2 counts per million** of the pseudobulk samples, with the standard error
and **one dot per donor sample**. Showing the dots matters: with few donors a bar alone hides how little data it holds.
The groups are **Control vs T2D** when a T2D group exists; otherwise the bars are **SAT vs VAT** and the title says so.

**Cell annotation:** Draw the bar graph (one panel per gene, free y-axis) for each depot.

In [ ]:
pb_cpm <- function(pb) {
  y <- DGEList(pb$counts); y <- tryCatch(normLibSizes(y), error = function(e) calcNormFactors(y))
  cpm(y, log = TRUE, prior.count = 1)
}
lc <- pb_cpm(pb_all)
g_present <- intersect(panel_genes, rownames(lc))
df <- do.call(rbind, lapply(g_present, function(g) data.frame(gene = g, sample = colnames(lc), logCPM = lc[g, ], pb_all$meta[colnames(lc), c("donor","depot","group")])))
df$gene_label <- factor(panel_names[df$gene], levels = panel_names[g_present])
if (mode_used == "T2D_vs_Control") { df$x <- df$group; df <- df[df$x %in% c("Control","T2D"), ]; ttl <- "Control vs T2D"; cols <- c(Control = "#4C78A8", T2D = "#E45756")
} else { df$x <- df$depot; ttl <- "SAT vs VAT (no T2D group in this dataset)"; cols <- c(SAT = "#4C78A8", VAT = "#F58518") }
sm <- df %>% group_by(gene_label, depot, x) %>% summarise(mean = mean(logCPM), se = sd(logCPM) / sqrt(n()), n = n(), .groups = "drop")
p <- ggplot(sm, aes(x, mean, fill = x)) + geom_col(width = 0.65) +
  geom_errorbar(aes(ymin = mean - se, ymax = mean + se), width = 0.2, na.rm = TRUE) +
  geom_jitter(data = df, aes(x, logCPM), inherit.aes = FALSE, width = 0.12, size = 1.6, colour = "black", alpha = 0.7) +
  facet_grid(gene_label ~ depot, scales = "free_y", switch = "y") + scale_fill_manual(values = cols) +
  labs(title = paste("Endothelial pseudobulk expression:", ttl), y = "log2 CPM (TMM-normalised); bars = mean +/- SE, dots = donor samples", x = NULL, fill = NULL) +
  theme_bw() + theme(strip.text.y.left = element_text(angle = 0), strip.placement = "outside", legend.position = "none")
print(p); ggsave(file.path(out_dir, "panel_bargraph.png"), p, width = 7, height = 15, dpi = 150)
write.csv(sm, file.path(out_dir, "panel_bargraph_values.csv"), row.names = FALSE)

*What you should see:* Eight rows (genes) by two columns (SAT, VAT). Bars with few dots mean few donors: interpret with the cautions in the results table.

## 11. Save everything and record the software

**Cell annotation:** Save the annotated objects (counts, metadata, cell types) and the software versions, so the analysis can be reproduced.

In [ ]:
saveRDS(obj, file.path(out_dir, "TS_fat_all_cells_annotated.rds"))
saveRDS(ec,  file.path(out_dir, "TS_fat_endothelial_annotated.rds"))
write.csv(obj@meta.data, file.path(out_dir, "metadata_all_cells.csv"))
write.csv(ec@meta.data,  file.path(out_dir, "metadata_endothelial.csv"))
writeLines(c(paste("date:", Sys.time()), paste("ambient:", amb_used), paste("analysis:", mode_used),
             paste("mito cut-off:", max_mito_pct), paste("h5ad md5 expected: 8269b64d360e060b36941a1261fcea0c"),
             capture.output(sessionInfo())), file.path(out_dir, "reproducibility_record.txt"))
list.files(out_dir)

## 12. Limits you should state when you present this
* **Tabula Sapiens is not a T2D cohort.** Unless the audit in section 3 found a T2D group with at least 3 donors, there is **no
  diabetes comparison** here. The SAT-vs-VAT result is a depot comparison in (presumably) non-diabetic donors and says nothing
  about T2D. For T2D you need GSE268904 / GSE278526 / GSE129363 (the other notebooks in this repository).
* **Few donors.** Tabula Sapiens fat has a small number of donors. Even a clean SAT-vs-VAT result rests on a handful of
  people, and unequal groups mean low power. A non-significant ERG result means "not detected with this sample size", not "no
  difference".
* **Marker lists were written from memory.** Check them against the Human Protein Atlas before you call a cluster.
* **10x and smart-seq2 are mixed** in this file. They differ in depth and chemistry, so `method` is a possible confounder;
  look at the MDS plot and the donor table. Integration is done by donor; the method is not regressed out.
* **ERG is lowly expressed**; many donor samples may have almost no ERG reads. The result table flags those.
* **Ambient correction changes counts.** The original counts are in assay `RAW`; rerun the pseudobulk on them
  (change `ambient <- "none"`) as a sensitivity check, as the guide suggests (contamination sensitivity).
* **Not done here:** leave-one-donor-out, composition tests (scCODA), pathways and TF activity (guide steps 14-17). They
  are worth adding once the first ERG result is stable.